In [ ]:
!pip install transformers torch matplotlib seaborn -q

In [ ]:
!pip install transformers torch matplotlib seaborn -q

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load saved model
MODEL_PATH = '../models/saved/bert_burnout'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, 
    output_attentions=True
)
model.eval()

print('Model loaded!')
print('Device:', 'GPU' if torch.cuda.is_available() else 'CPU')

In [ ]:
def get_attention(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get predicted label
    predicted_id = outputs.logits.argmax().item()
    predicted_label = model.config.id2label[predicted_id]
    confidence = torch.softmax(outputs.logits, dim=1).max().item()
    
    # Get attention weights - last layer, average all heads
    attentions = outputs.attentions[-1]  # last layer
    avg_attention = attentions.mean(dim=1).squeeze()  # average heads
    
    # Get tokens
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    # CLS token attention (what each word contributed)
    cls_attention = avg_attention[0].numpy()
    
    return tokens, cls_attention, predicted_label, confidence

# Test it
text = "I feel so exhausted and hopeless, I can't continue my studies anymore"
tokens, attention, label, confidence = get_attention(text)

print(f'Prediction: {label} ({confidence:.2%})')
print(f'Tokens: {tokens}')
print(f'Attention shape: {attention.shape}')